# WEL01 · M02 전처리 전후의 오차와 처리 속도 확인 (교수자·해설용)

- 본 노트북은 **미션2(M02) 전용 교수자·해설용** 노트북입니다 (`ROLE = 'solution'`).
- 원자료 생성·참고 구현·상황판 소스가 노트북에 내장되어 있어 별도의 `.md` 파일 없이 위에서부터 바로 실행할 수 있습니다.
- 각 STEP 코드 셀에는 모범 pandas 풀이, 경계 조건 예외 처리, 해설 주석이 포함되어 있습니다.

In [ ]:
from pathlib import Path
import base64,zlib,json,re,html,hashlib,io,time,ast,copy
import numpy as np,pandas as pd
from IPython.display import display,HTML

PROJECT_ID='WEL01'
ROLE='solution'
WORKDIR=Path('/content') if Path('/content').is_dir() else Path.cwd()
PROJECT_HOME=WORKDIR/PROJECT_ID
OUTDIR=PROJECT_HOME/'outputs'
SOURCE_BUNDLE=None
DASHBOARD_ASSETS=None
REFERENCE_CACHE={}
REFERENCE_EXECUTION_SERIAL=0

def _sha(value):
    return hashlib.sha256(json.dumps(value,ensure_ascii=False,sort_keys=True,separators=(',',':')).encode()).hexdigest()

EMBEDDED_BUNDLE_B64 = '...'
EMBEDDED_ASSETS_B64 = 'eNqNOl1z20aSfwVxNgERgwgpiaIEHuLzeu0938ZOKvLePsgqGQSGJCIQ4GJASjqGW/Za2XIib61u177IieSS7+zE3kvqFH8qVd6X/BwC+g/XPTMAAYrM5cEiMNPT3dPf3XD/lEXpKf3UO/26v1Gkzr87XlOv+4FNgiKsDOq+vdlvm0HT8fRSrW5aa83A73q2/iZpNGYb8zXLd/1Af7NcnanPWrWG74V6udLZeLesVSrS2cAxXVW+7Ie+tGR6VPrNR7JK4aFISeA0BvVuGPqe6nidbqiGZCM0A2L2GRLHawFIKED6VjegQKfjO15IgoHmmpt+N+zbDu3Ao94MHLuGf4ohacNKSIrAWLftUb08O9PZkNqO1zY3CiW13AiUGrwVW8RptoDZUqnXGmg0JJ2iZ/b62SuWq7P1uUVxxfWWE5Jax7RtFFF5AZAudjZqHZ86oeN7Og0da22zFvodEJTpOk0PLuk2YN0MwtpkahINA99rpteou761xoSIqiCMSkqyJCHNmVJnI4NAiCePYN2xwxbSequGQi0ybnSXNMJaokqpJJXLGeRlFFIZcNe48nXYlKjvOrb05lx93qxWxEYxMG2nS/Uqgo5kFQag1Q6ozwuz8jrB6bIJJlEEbSKkgZsrOZGTcn1h3kytqjRjzy4mlMUiB8kKsW267nQZ4jVdxyOpxrU5IYYiKquC8rQAHBjqo2Fw6ZVS0cwAhDSLogELErvlSonpgWySeuCv9wVrc4vzi9WFLPEZJE5CsNkiiMdCfFppgbRTklKr3B/BI60xZmcridJA5Ki2UoZhqTWTPc2ZTDW8kIOczUCW5ydAUsvvkOQq89U5u7yY8/hGo1Gx6yOTKaHJzKUmk1hGBjVGAqk00NAzf46vjrxUq4KfJl47Mw+kuOc2zQ73iowGuUt0TI+4OQVmeOfOe9K07SopkXHTXsz6xeKIVt0HC24z8gMtNOsuKVIr8MH2/B4JGq6/rpvd0GdmItQ3V0HeGGx/ZMWu2aFETx6y7pq3nEHYUkO7P4Fti8yRhRPOnTBdFbGJ3ZrZHdE9fz0wO4Ax724LjVJjdnIQG4QY/KUw0L2wVbRajmsXSI94Sg5Do9owG3WQh++7dTNItdxwyQbTFiqLh0Ngpk11i2AEr+F+EVnS8U9qMDPcYAS2JLyNWeEERdbnrarVGFNkJaPICpfJCDVLOv3s9sLE6CcwCyXNsXDRJmHgWD8r+zQCiRlFIoo0t3JbGtEhM2SRWDkdlrL2MDsiLNVTi+MiNL3NdUiXJGsRARognOiGxO6PRSTh45V6tVy1wHVcP+xnrFAYL+M7H1YzXldmknD9Zj9rZp2AMJZqUxgci8sZV5lFV6nlPGmgeX5IspxlEzfLVgFhyOAUWK7p/lzLmMtYBo99oRl2aRL8ZiqV6lyVM7vOydV91x5oDR8kNXaHJGAuVK2F6kkJUbNH/r+qQnBd+gn7xdzPYsI/t4ntmIVRLlpAwSn9NIdlSxQReqfaZlpITYFgMT5XO2VKpZQQOs/8pLIklyqquWyUy3ogxsHgqnfV0xwPSqKuhcGIZgV9Mi8uVKbY2CCHRGrNJfXrzBxLoEwsORAen8d9LQdi+Tbp5+NpAGXHiSpkrNCYzmMqxRRgU9j8BNZ4suGq0hvOBrHHwVrq2II9gdtpzFz1TqmnPsY2oFBQjPf6cpcSLE0dK5TBTgGvZBnrjmf769rvzr9fKq+e++DyhYu/xspGMoFijxgltWM28afhuBDfDTk5iS5MjbNBYG4WKooG225BlhWxGzpgXU1qLK+IBeIahdBsqpZLWUeA/PCdDcP2rW4brEezoE0IyXmX4BuCKzWnUYAjyoZmuSall802MeAdlxHLG4YBvkcaoB0bYHDpHLdEYwnu6TUZlALhJOwGnrQxENyYnc6IapOEguQvNy/aBRk203tw3RgE7mY7PVmV+UK6D5pmm/ALm4nyYRv+apSEZ0NgA7yGAFqskF2zTlyAjP73KP50V4q2Hw+fbQlwoEs8yMWAjvcPsup1XVe1tLZDKahfSai2TcdjZPEBsAnvAzycvQQVYFURRKnBQrLIQRJU3KOZoq56ja7H7EwCi/Bc30QMbaL2TLdLFKEuc5q6ZBPom1orIA3jtx+9L3Y/qH9MrBDeCx5Zl37p+vXC8r8ufXBZo0w/TmOzwNDzq84oK2o/3OwQHbXgQuhHdt79mPqePFAQfcKZgZzBuwVAawWlBrK+4rQJ3IzZOjIQkJ6/lmGA86aokHFKyiBzW9ry16+gKxZ4p6OKOKlCC0CTe6NrZewgWyeOrB4XGRB7StdbBBhmy/gkK2oY8NeAgTBikIOC86bVKqwZ74VB1hjCljCENQVEgBiS3RAqZ0YpWcDNVLNQ5XEq+ASEgECBbwWS35Cyl4PnEUMjuDWEE/wloD0jWF5bqcGJHI+24LFnGAY+fPIJPqXOeQYt/vje3Xj/SNZRwYAYAWSfqUc+M24Tit5TFDamSMgARWWQuy3uKjVUTCoQ3FZqXI3JIgLk9M2rrUTZa2pvXAxcxRwMBDJ2Vwi9XqoRFVfqyeWVcdqM6Qxp2INCoJAQpIaFJUqT0GUeb1dqwiFTcyjUVQeDJUQ8BwTGwZT6pOAi2m8RiIBx4kLAr4MjtCFDTINWBkoNgwSAQZVgkXPYEcBOQSxnbs7lIppjWb32i76ldQIfNXjRHkg/vpJwRUSr9L3ldwM6iLf3hodbuBZvP4wP9qQPN8MWSATejv/zM2n49HD47PU1Ls9WOQ19oROCRtlqJ12k0GzAHXxlGous5YXf4YudeO9B/OUT6fjuk3jreym+vxP9923GxTdbEH2l+C787LKFw6PhswMp+vvj6M9Hgi1gCU7eiz9/AQfj/V0JVG+bVHAbPXotDV98G73cAuzR9q1o+6GGC/GDQ4C4Hn+9H+9fl+IXe/F/fSp9eHZpSTq+vRMfgBM853TviUNyGtmxXsuYIL5CuMAuMF1VVNYCjN7FUdYos1VKmLHBebYmo1HCr2aboQlmo3EbWgIbSbIkXzldTiAzAm3NCLFfW7py/kNQ6PgR6NDtJRyGFWZUuQTmJAEQ5YobCIUKh8KehZkNBSPx4I0OQHhSdHgIUmE6uLnPtPTFc4YDojk0c+dYAGLmFH+1g8aCe2CB56DwDgfDwz0pevY8fvD8msLqBaq5xAw8EvxL2HYTT8tUUWMyytZXIKrMK5ReiOXKpfeNHM68kDIHlAEnxmslYhuMTarx90JgvPcGf/zkEyHFYFnmd7r4K3lFAYKW27UJLXCw1CywuaXZ7MObXbAFaHfZOmt7GfsdjblxCxobrNcS9Mwib34mM4ifqEzG4TlKlqMNzhVb8D2rZXpQGbLKUhSHKWSNF421JNwldRe0kbxe4lFOWJYMDhEfbElAF27kQcU2AQb8JN6/zWFqiEeDFhbDvW0wYgaQw6Oj5QKug4G+M1t6z0hUAnr0mmGLY4BLYPnA74DQxWKGZYbtBMjp0xkQppeRIXRURMtuoI4yRcbuEQGwc7o8+MMv+pfMsIV+UMgwqo7xCd70Lpj72OoApHBNGXNWxgsUQpliBnZVqiXVTIqEwo1IQfCiZqgrE/y/w9IJdtGpcfz4Cn6jZ0fDw7/++Cq6tRUf3BARkXluLgJK8b0tiLgY4TCKJZgZmcS6KSQIi0wJXXwzF5FmU7v5akcE9Oj2daQpxU//R5pnNpKVBMeBiYM9nBMVF5dDSV1U0q2P0F2T9fkxpjmIkmmApvCMW5M5Hj6/Hh9ySe1DutmShkeH0YP9UQJIvpbwcky8AGpECZiTlTEf3zo4vrl/fHdPGr56Pfz+UGXp6/Zj0E5SdUnR0zvx3pYqJX7EVpGRaPt5tP1DvJ/wcnx3N946ir+8o8kjctz/Wc+XVirppu+x6MN8JAdi5M/n5JJspUZg9kjO7WVcwRzOuJKwQJTig+vx/UdoFbCX8820Y7nGO9lsIbIaEMsPbA3biGtqP61Y9Ez1ogpgPT2mcnVfgDBNgg5E6xD2GqM3lV1V7TqXiEm7AWuCqC4aX7VNTA9HJHJaPkhc90LKEPKiJ4dgBbtovuA9kAF/fAX1CkCJMkHUJtD45ASHV8/bJe6mYuy222awOc2Z+O4U20yLpfjmjeN7T4ALtEtRLYujqny8uxXf/zZpXTGqJaUAxqrKtZMnRm6Knpn3NRHmTpwZxRNZzRcCE/Bz5Lx4kFk6iZ7eAhPfHT79R/zFTry1l1R38T/uRH/bE9WdPAGVAONFafz8lqxe7rbrEO0tHNKaUCQ3IIZ6FlkF6fqeTRUt9C/g6KYwq4ymBpBLqdE3Pb9tupurFhYqOrjpa8YKZLubN4DG3vEXu1J8axdEAg08dJIpINSIACGCWhawGZg2WUUqDtT4nrWpy9Hfvh8eAa/7r9HAUAABcU3UyGrDDMD2bV6c6PKFsx9Jw8Od+OXd6NEBgDl0DV3DdF0gubfFiNz/FvW791hW2Twq3QerjR9CMXsCrNuBspKsor2vtqkui7J+ePh9vH0gHf/ls+GrA6lNR4DmxgQ4KJLhtgyOZ1cGI7QxfHUUfQ2B6uB29Oc70Tc3GBgbXAODicgOH4NFRjsg3i/vxE+/4+Iyu7YTrkLwcezVOoGWioC8nnGpQsERPdgD3CJTyWpotjtA2SYhOA6xU1Dcf7mX7qdfVVJUe1vxD0CtUV5tQjuy2umArMtwtT0IzNJbkEEtaBscdn3B7vHNHSiupOOvtqB0TWygRwJIxXCXb66j+x/ffQwbiS1GP3B5Wy1iraW3hq7kKZw/2AEe4cB30d+fcGwdk1K8whjEn14wr0IIMI7U0iBlY3MGBiKnpRoJ8JOOgWObJRIWluUTtgeuNtnUcGNkXPCWtSU2r+JXxR2oYleyM4flNZW5zwrOHvjwRgM+AgeqYu5XymgUwZ0S5xFQ9+cGDnz+gIFNQeeCboMkERLKAs8Q19NaJi2sKWcKwst7yjvlUmnk02XltPxWfmLhMUj57bff4Gc0h170QtJkp8/00qMzSu5YHcoz8BH5TKF3Ro53b8kgfIzvt+Ldh7Ki98ZDEburyphVBhPidmdUWIrQhO3dKut8IRCxHjFNGfx4miMQZlqGwL0p+YEXdLyQiO9dl6I/PsYGV4x10iZ9jK22bxN31eyCuwe05XSU3ISJbKKel2X8/x9gYMjNqgs5FKcYzG/5UtLQi7SKLRBLtKtdj4EmFpUCcs9nVdOKgkNkYSlkc0U5cccTTHO4CdLIQMq/vSiSIMYSSDYSH3Gwnp+POI5vPI++/g69XIqPHoCvQW2FWT9T0kB6ElVNZoaQS1Y4dohf3okeHEB9IKIlI8SLt4fRI4hgh3+N73+amSbk1c9sopZKQWMxhL79du41ScX9pBClIZ1iJGxvSkmeJFCesyDzb++dKHKhS2KY4RdH6j7oc2z+l+MsmQ2PEYbTYxdl28oApybptBvqJD4wyc+KEEaIBD+Tjt+erU0bpjD4n749huyk8mBlCS/OMY/JOQeggcWGrJyLdFrRbvKuvo2mjlvYh7ebGoAb8I89m25oJNVYnjBYTDLv2j443oXif3cn+vyOBCvR548SPvIXAYzKICshzpCYaICOpkgDdibLIs8ZZBhYSJMZS7xJyBhZgVA7DW38RJHXGACkEQa/10LpeelXEuTK6NsjNmfM1OaDzJwRXCt69h/QCr28jiUGeCFLuV/8CdogbKJzH8D+7eL5360uXTl75bzRP9kRjAZnelLzqskcSh+NpNRsuarnq1coqwIaYumri+FQ0m3OKlN6iUENJ8hgMfhZzjFKNeefKjXn9OnEWOrZtkmp1dM5H81M+Bwc7tWnjLHHRnvOpKmemOSxz6GjQSyfXTsrYk6LJHITExFLMN028FslFM6a568XlJr4vugko6LR98VkvAIPv++CR5/1nDbLAhcCs03Y553pO30hNK3Tpa1CH0Wgw32Ad/w/OfYlqp9gpRiqLELqMmO8GPrFcN0vmgnyYgOxU+zDptiKlleckShOUQbpPJ/xU8996asrg/SuAwX+XPVODf4PT6yS7w=='

def _decode_b64(payload):
    return json.loads(zlib.decompress(base64.b64decode(payload, validate=True)))

def load_embedded_bundle():
    global SOURCE_BUNDLE, DASHBOARD_ASSETS, REFERENCE_CACHE, REFERENCE_EXECUTION_SERIAL
    bundle = _decode_b64(EMBEDDED_BUNDLE_B64)
    loaded_assets = _decode_b64(EMBEDDED_ASSETS_B64)
    if bundle.get('projectId') != PROJECT_ID or bundle.get('version') != 'wel01-integrated-2':
        raise ValueError('프로젝트·버전 불일치')
    if bundle.get('fingerprint') != _sha({k: v for k, v in bundle.items() if k != 'fingerprint'}):
        raise ValueError('원자료 지문 불일치')
    if bundle.get('assetFingerprint') != _sha(loaded_assets):
        raise ValueError('화면 소스 지문 불일치')
    for m_id, item in bundle['missions'].items():
        if len(item['stages']) != 5:
            raise ValueError(m_id + ' 5단계 누락')
        if sum(s['minutes'] for s in item['stages']) != item['minutes']:
            raise ValueError(m_id + ' 시간 불일치')
        if hashlib.sha256(item['referenceCode'].encode()).hexdigest() != item['referenceCodeSha256']:
            raise ValueError(m_id + ' 참고 코드 지문 불일치')
    namespace = {}
    exec(compile(bundle['runtimeSource'], 'WEL01_reference_runtime.py', 'exec'), namespace)
    bundle['_reference_runner'] = namespace['_reference_table']
    SOURCE_BUNDLE, DASHBOARD_ASSETS, REFERENCE_CACHE = bundle, loaded_assets, {}
    REFERENCE_EXECUTION_SERIAL = 0
    print(f'WEL01 M02 내장 교육 자료 연결 완료 (5단계 · {bundle["missions"]["M02"]["minutes"]}분)')
    return bundle

def _connected():
    if SOURCE_BUNDLE is None:raise RuntimeError('먼저 첫 실행 셀을 실행해 내장 교육 자료를 로드하세요.')

def _parameter_code(mid,code,parameters):
    allowed=SOURCE_BUNDLE['missions'][mid]['parameters']
    if set(parameters)-set(allowed):raise ValueError('정의되지 않은 변경 파라미터')
    if mid=='M02':
        value=parameters['FLATLINE_RUN']
        if not isinstance(value,int) or value<1:raise ValueError('고착 분은 양의 정수')
        code=code.replace('def flatline(series, run=8):',f'def flatline(series, run={value}):')
    elif mid=='M05':
        value=float(parameters['SCORE_THRESHOLD'])
        if not 0<=value<=1:raise ValueError('문턱은 0~1')
        code=code.replace('eng = (test["score"] >= 0.5).values',f'eng = (test["score"] >= {value!r}).values')
    elif mid=='M09':
        value=float(parameters['MIN_RECALL'])
        if not 0<=value<=1:raise ValueError('재현율 하한은 0~1')
        code=code.replace('(g["재현율"] >= 0.90)',f'(g["재현율"] >= {value!r})')
    inserts={}
    for node in ast.parse(code).body:
        if isinstance(node,ast.Assign):
            names=[]
            for target in node.targets:
                names.extend(x.id for x in ast.walk(target) if isinstance(x,ast.Name))
            updates=[f'{key} = {parameters[key]!r}' for key in names if key in parameters and key not in ['MIN_RECALL','SCORE_THRESHOLD','FLATLINE_RUN']]
            if updates:inserts[node.end_lineno]='\n'.join(updates)
    lines=code.splitlines()
    for pos,value in sorted(inserts.items(),reverse=True):lines.insert(pos,value)
    return '\n'.join(lines)+'\n'

def source_table(mid,parameters=None,refresh=False):
    global REFERENCE_EXECUTION_SERIAL
    _connected();item=SOURCE_BUNDLE['missions'][mid]
    p=dict(item['parameters']);p.update(parameters or {})
    key=(mid,_sha(p))
    if refresh or key not in REFERENCE_CACHE:
        tuned=None
        if mid=='M10':
            # 현재 M9 실행 결과를 우선 사용  아직 실행하지 않았으면 기본 M9를 실행한다.
            matches=[(k,v) for k,v in REFERENCE_CACHE.items() if k[0]=='M09']
            if not matches:
                source_table('M09');matches=[(k,v) for k,v in REFERENCE_CACHE.items() if k[0]=='M09']
            m9=max(matches,key=lambda x:x[1][1]['executionOrder'])[1][1]['selected']
            tuned={'heat_temp':float(m9['th_heat']),'cold_temp':float(m9['th_cold']),
                'min_persist':int(m9['persist']),'wind_corr':float(m9['corr'])}
        code=_parameter_code(mid,item['referenceCode'],p)
        raw,meta=SOURCE_BUNDLE['_reference_runner'](mid,code,tuned)
        REFERENCE_EXECUTION_SERIAL+=1
        meta['executionOrder']=REFERENCE_EXECUTION_SERIAL
        meta.update(parameters=p,md_fingerprint=SOURCE_BUNDLE['fingerprint'],reference_name=mid+' 교육용 참고 구현')
        REFERENCE_CACHE[key]=(raw,meta)
        if mid=='M09':
            for stale in [k for k in REFERENCE_CACHE if k[0]=='M10']:del REFERENCE_CACHE[stale]
    raw,meta=REFERENCE_CACHE[key]
    out=raw.copy(deep=True);out.attrs['reference_meta']=copy.deepcopy(meta)
    return out

def TODO(mission,step,instruction):
    raise NotImplementedError(f'{mission} STEP {step}: TODO를 실제 pandas 코드로 완성하세요. {instruction}')

def _rows(df):
    return json.loads(df.to_json(orient='records',force_ascii=False,date_format='iso'))

def finish_mission(mid,raw,stages,stage_specs):
    _connected();canonical=SOURCE_BUNDLE['missions'][mid]['stages']
    if stage_specs!=canonical:raise ValueError(mid+' 내장 기준과 노트북 STEP 정의 불일치')
    if len(stages)!=5 or not isinstance(raw,pd.DataFrame) or raw.empty:raise ValueError('5단계와 원자료 필요')
    expected=list(raw.columns);original_ids=raw['원행ID'].tolist();report=[];cumulative=[]
    for k,(df,spec) in enumerate(zip(stages,stage_specs),1):
        if not isinstance(df,pd.DataFrame):raise TypeError(f'{mid} STEP{k}: DataFrame 필요')
        col=spec['outputColumn'];cumulative.append(col)
        if df['원행ID'].tolist()!=original_ids or not df.index.equals(raw.index):
            raise ValueError(f'{mid} STEP{k}: 행 누락·재정렬·중복이 발생했습니다.')
        if any(x not in df.columns for x in expected+cumulative):raise ValueError(f'{mid} STEP{k}: 필수 열 누락')
        if not df[expected].equals(raw[expected]):raise ValueError(f'{mid} STEP{k}: 원자료를 변경했습니다.')
        if df.columns.duplicated().any():raise ValueError('열 이름 중복')
        preview=df[['원행ID']+cumulative]
        report.append(dict(step=k,title=spec['title'],goal=spec['goal'],outputColumn=col,
            learnerHtml=spec.get('learnerHtml',''),rowCount=len(df),columns=list(preview.columns),rows=_rows(preview),minutes=spec['minutes']))
    location=OUTDIR/mid;location.mkdir(parents=True,exist_ok=True);stem=f'WEL01_{mid}'
    stages[-1].to_csv(location/(stem+'_analysis.csv'),index=False,encoding='utf-8-sig')
    meta=copy.deepcopy(raw.attrs['reference_meta']);plots=meta.pop('plots',[])
    details=dict(projectId='WEL01',mission=mid,originalRows=len(raw),codeSteps=report,
        referenceMeta=meta,mdFingerprint=SOURCE_BUNDLE['fingerprint'],
        structuralValidation='원행과 원본값·누적 열 검증 완료  코드의 의미·교관 PASS 판정과 별개')
    (location/(stem+'_code_report.json')).write_text(json.dumps(details,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
    config=dict(projectId='WEL01',mission=mid,title=SOURCE_BUNDLE['missions'][mid]['title'],
        scenario=SOURCE_BUNDLE['missions'][mid]['scenario'],hours=SOURCE_BUNDLE['missions'][mid]['minutes']/60,
        stages=report,sourceColumns=list(raw.columns),sourceRows=_rows(raw),meta=meta,plots=plots,
        fingerprint=SOURCE_BUNDLE['fingerprint'],role=ROLE)
    safe=json.dumps(config,ensure_ascii=False,default=str).replace('<','\\u003c')
    document='<!doctype html><html lang="ko"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>'+stem+'</title><style>'+DASHBOARD_ASSETS['css']+'</style></head><body><div id="app"></div><script>window.WEL01_CONFIG='+safe+';</script><script>'+DASHBOARD_ASSETS['js']+'</script></body></html>'
    path=location/(stem+'_result.html');path.write_text(document,encoding='utf-8')
    if not globals().get('HEADLESS_VALIDATION',False):
        display(HTML('<iframe title="'+stem+'" sandbox="allow-scripts allow-downloads" style="width:100%;height:1150px;border:0" srcdoc="'+html.escape(document,quote=True)+'"></iframe>'))
    print('HTML:',path);print('행 보존·누적 열 검사 완료  최종 PASS는 교관 판정')
    return path

load_embedded_bundle()

## M02 전처리 전후의 오차와 처리 속도 확인

신호를 정리하는 과정에서 오차가 줄었는지, 실제로 충분한 양을 처리했는지 확인해야 합니다. 1분 간격 표본 8개가 나타내는 시간은 7분입니다. 오차와 속도, 단위 시험, 8분 고착의 시간 경계를 따로 계산한 뒤 어떤 항목을 보완해야 하는지 설명하세요.

실습 시간 120분. 준비 셀 → STEP별 설명과 코드 → 마무리 셀 순서로 실행합니다.


In [ ]:
MID = 'M02'
PARAMETERS = {'FLATLINE_RUN': 8}
raw = source_table(MID, parameters=PARAMETERS, refresh=True)
print("입력 표의 행·열 수:", raw.shape)
display(raw.head(6))


### M02 STEP 01 — 전처리 후 체온 오차가 얼마나 줄었는지 계산하기

<h4>이 단계에서 해결할 일</h4><p>센서에서 받은 값을 정리했다고 해서 실제 기준값과의 차이가 줄어드는 것은 아닙니다. 평시·폭염·혹한의 각 상황에서 전처리 전후 오차를 비교해야 합니다. RMSE는 여러 시점의 측정 오차를 하나의 수로 요약한 값이며, 여기서는 작을수록 가상 기준 체온에 가깝습니다.</p><h4>시작 전 준비</h4><p>각 미션의 STEP 1~5는 앞 단계 결과를 이어서 사용합니다. WEL01_m02_student.ipynb의 첫 실행(환경 준비) 셀과 이 미션의 준비 셀을 순서대로 실행하면 현재 참고 프로그램의 결과가 raw라는 입력 표로 만들어집니다. PARAMETERS는 기본값을 유지합니다. 이 단계만 다시 실행하려면 그 미션의 준비 셀부터 바로 앞 STEP까지 순서대로 실행해 둡니다.</p><p>이번 입력 표는 <code>raw</code>, 완성한 결과 표는 <code>step1</code>입니다. 새로 만들 열은 <code>RMSE_개선량</code>이며 배정 시간은 24분입니다. 노트북의 <code>M02 STEP 01</code> 코드 셀에서 TODO 호출을 본인이 작성한 pandas 계산으로 바꾸세요. 결과는 DataFrame이어야 합니다.</p><h4>입력값의 뜻</h4><ul><li><code>체온 RMSE 전</code> — 센서 결측·튀는 값·고착을 정리하기 전의 체온 오차입니다.</li><li><code>체온 RMSE 후</code> — 같은 상황 자료에 전처리를 적용한 뒤의 체온 오차입니다. 단위는 ℃입니다.</li></ul><h4>수행 순서</h4><ol><li>raw에서 상황 열을 보고 평시·폭염·혹한의 3행이 있는지 확인합니다.</li><li>각 행에서 체온 RMSE 전에서 체온 RMSE 후를 뺍니다. 서로 다른 상황의 값을 섞지 않습니다.</li><li>차이를 RMSE_개선량 열에 추가하고 step1에 저장합니다. 단위는 두 입력과 같은 ℃입니다.</li><li>양수이면 오차 감소, 0이면 변화 없음, 음수이면 오차 증가라고 상황별로 기록합니다.</li></ol><h4>결과를 읽는 방법</h4><p>개선량이 양수라고 해야 오차가 줄었다고 말할 수 있습니다. 이 값은 합성 자료의 기준 체온에 대한 오차이며 실제 진단 성능이나 경보 성능을 뜻하지 않습니다.</p><h4>직접 확인할 가상 예시</h4><p>다음은 규칙을 이해하기 위한 가상 사례입니다. 실제 실행값을 대신 제출하는 자료가 아닙니다.</p><table><thead><tr><th>가정한 입력</th><th>기대 결과</th><th>이유</th></tr></thead><tbody><tr><td><code>{"체온 RMSE 전": 0.146, "체온 RMSE 후": 0.104}</code></td><td><code>0.042</code></td><td>오차가 0.042℃ 줄었습니다.</td></tr><tr><td><code>{"체온 RMSE 전": 0.1, "체온 RMSE 후": 0.12}</code></td><td><code>-0.02</code></td><td>오차가 0.02℃ 늘었으므로 개선이라고 쓰면 안 됩니다.</td></tr></tbody></table><h4>완료 전 확인과 기록</h4><p>원행ID는 계산 전후 같은 자료를 찾는 번호입니다. 입력의 행 수·행 순서·인덱스·원래 열의 값을 유지하고, 앞 단계에서 만든 열도 모두 남깁니다. 각 행의 계산 결과는 새 열에 넣습니다. 실제 불리언 True·False와 문자열 "True"·"False"는 다릅니다. 값이 없다는 뜻의 결측값을 False나 0으로 대신 채우지 않습니다.</p><p>계산 뒤 해당 출력 열과 입력 열을 원행ID로 함께 확인합니다. 실제 자료에서 해당 사례가 있으면 최소 한 행의 입력·출력·판단 이유를 기록하고, 해당 사례가 없으면 0건 집계와 아래 가상 예시 시험 결과를 남깁니다. 관찰 내용은 5단계 완료 후 HTML의 해당 STEP 기록란에 옮기고 기록 JSON 저장을 눌러 보관합니다. HTML 입력란의 글은 페이지를 닫기 전에 저장해야 합니다.</p><h4>주의할 점</h4><p>나중에 경보 F1과 오경보 건수도 함께 확인합니다. 체온 오차가 줄었다는 이유만으로 위험을 덜 놓치게 되었다고 단정하지 않습니다.</p>


> **교수자·해설용 FAIL 점검 포인트**: 후-전으로 빼서 부호를 뒤집거나, 절댓값을 취해 악화 사례를 개선으로 바꾸면 FAIL입니다.

In [ ]:
# M02 STEP 01 — 전처리 후 체온 오차가 얼마나 줄었는지 계산하기 (교수자·해설용 예시 풀이)
# 해설: 개선량은 (체온 RMSE 전 - 체온 RMSE 후)입니다. 음수(오차 증가)도 절댓값 없이 부호를 유지합니다.
step1 = raw.copy()
step1["RMSE_개선량"] = step1["체온 RMSE 전"] - step1["체온 RMSE 후"]
display(step1[["원행ID", "RMSE_개선량"]].head(10))

### M02 STEP 02 — 실제로 처리한 신호값 수로 초당 처리량 구하기

<h4>이 단계에서 해결할 일</h4><p>전처리가 충분히 빠른지 확인하려면 실제 처리한 양을 실제 걸린 시간으로 나누어야 합니다. 이번 예시 프로그램은 한 시점의 심박·체온·피부전도 값을 각각 한 개로 셉니다. 따라서 표 한 행을 처리한 수와 신호값 세 개를 처리한 수는 다릅니다.</p><h4>시작 전 준비</h4><p>각 미션의 STEP 1~5는 앞 단계 결과를 이어서 사용합니다. WEL01_m02_student.ipynb의 첫 실행(환경 준비) 셀과 이 미션의 준비 셀을 순서대로 실행하면 현재 참고 프로그램의 결과가 raw라는 입력 표로 만들어집니다. PARAMETERS는 기본값을 유지합니다. 이 단계만 다시 실행하려면 그 미션의 준비 셀부터 바로 앞 STEP까지 순서대로 실행해 둡니다.</p><p>이번 입력 표는 <code>step1</code>, 완성한 결과 표는 <code>step2</code>입니다. 새로 만들 열은 <code>신호값_초당처리</code>이며 배정 시간은 24분입니다. 노트북의 <code>M02 STEP 02</code> 코드 셀에서 TODO 호출을 본인이 작성한 pandas 계산으로 바꾸세요. 결과는 DataFrame이어야 합니다.</p><h4>입력값의 뜻</h4><ul><li><code>처리신호값수</code> — 심박·체온·피부전도 등 실제 처리한 개별 신호값의 합계입니다.</li><li><code>실행초</code> — 해당 전처리 구간에 실제로 걸린 시간입니다. 단위는 초입니다.</li></ul><h4>수행 순서</h4><ol><li>step1에서 처리신호값수가 무엇을 세는지 확인하고 실습 실행 기록의 단위와 맞춥니다.</li><li>실행초가 0보다 큰 수인지 확인합니다. 읽기·학습·그림 생성 시간이 아니라 기록된 전처리 시간을 사용합니다.</li><li>처리신호값수를 실행초로 나누어 신호값_초당처리 열에 추가하고 step2에 저장합니다.</li><li>보고할 때 개별 신호값/초라고 단위를 적습니다. 1000 이상이면 이 실습의 처리량 목표를 충족한 것입니다.</li></ol><h4>결과를 읽는 방법</h4><p>신호값_초당처리는 현재 실행 환경의 처리량입니다. 1000개/초 이상인지 확인하되, 다른 컴퓨터에서도 같은 시간이 나올 것이라고 가정하지 않습니다.</p><h4>직접 확인할 가상 예시</h4><p>다음은 규칙을 이해하기 위한 가상 사례입니다. 실제 실행값을 대신 제출하는 자료가 아닙니다.</p><table><thead><tr><th>가정한 입력</th><th>기대 결과</th><th>이유</th></tr></thead><tbody><tr><td><code>{"처리신호값수": 21600, "실행초": 2}</code></td><td><code>10800</code></td><td>초당 신호값 10800개입니다.</td></tr><tr><td><code>{"처리신호값수": 3000, "실행초": 4}</code></td><td><code>750</code></td><td>750개/초이므로 처리량 목표에는 못 미칩니다.</td></tr></tbody></table><h4>완료 전 확인과 기록</h4><p>원행ID는 계산 전후 같은 자료를 찾는 번호입니다. 입력의 행 수·행 순서·인덱스·원래 열의 값을 유지하고, 앞 단계에서 만든 열도 모두 남깁니다. 각 행의 계산 결과는 새 열에 넣습니다. 실제 불리언 True·False와 문자열 "True"·"False"는 다릅니다. 값이 없다는 뜻의 결측값을 False나 0으로 대신 채우지 않습니다.</p><p>계산 뒤 해당 출력 열과 입력 열을 원행ID로 함께 확인합니다. 실제 자료에서 해당 사례가 있으면 최소 한 행의 입력·출력·판단 이유를 기록하고, 해당 사례가 없으면 0건 집계와 아래 가상 예시 시험 결과를 남깁니다. 관찰 내용은 5단계 완료 후 HTML의 해당 STEP 기록란에 옮기고 기록 JSON 저장을 눌러 보관합니다. HTML 입력란의 글은 페이지를 닫기 전에 저장해야 합니다.</p><h4>주의할 점</h4><p>실행초가 0이거나 비어 있으면 계산할 수 없습니다. 0으로 나누거나 임의의 작은 시간으로 바꾸지 말고 실행 환경과 재측정 필요 사항을 기록합니다.</p>


> **교수자·해설용 FAIL 점검 포인트**: 초를 밀리초로 오해하거나, 입력 행 수로 분자를 바꾸거나, 예시 시간을 복사하여 처리량을 계산하면 FAIL입니다.

In [ ]:
# M02 STEP 02 — 실제로 처리한 신호값 수로 초당 처리량 구하기 (교수자·해설용 예시 풀이)
# 해설: 실행초가 0 이하이거나 비어 있으면 0으로 나누지 않고 예외 처리합니다.
step2 = step1.copy()
if step2["실행초"].isna().any() or (step2["실행초"] <= 0).any():
    raise ValueError("실행초가 0 이하이거나 결측입니다.")
step2["신호값_초당처리"] = step2["처리신호값수"] / step2["실행초"]
display(step2[["원행ID", "신호값_초당처리"]].head(10))

### M02 STEP 03 — 단위 시험 결과와 처리 중 오류를 함께 확인하기

<h4>이 단계에서 해결할 일</h4><p>일부 시험만 통과했거나 실행 중 오류가 발생했는데 전처리 전체가 정상이라고 보고하면 안 됩니다. 개별 기능을 확인한 단위 시험 결과와 많은 자료를 처리하면서 발생한 오류 수를 함께 확인합니다.</p><h4>시작 전 준비</h4><p>각 미션의 STEP 1~5는 앞 단계 결과를 이어서 사용합니다. WEL01_m02_student.ipynb의 첫 실행(환경 준비) 셀과 이 미션의 준비 셀을 순서대로 실행하면 현재 참고 프로그램의 결과가 raw라는 입력 표로 만들어집니다. PARAMETERS는 기본값을 유지합니다. 이 단계만 다시 실행하려면 그 미션의 준비 셀부터 바로 앞 STEP까지 순서대로 실행해 둡니다.</p><p>이번 입력 표는 <code>step2</code>, 완성한 결과 표는 <code>step3</code>입니다. 새로 만들 열은 <code>단위시험_충족</code>이며 배정 시간은 24분입니다. 노트북의 <code>M02 STEP 03</code> 코드 셀에서 TODO 호출을 본인이 작성한 pandas 계산으로 바꾸세요. 결과는 DataFrame이어야 합니다.</p><h4>입력값의 뜻</h4><ul><li><code>통과테스트수</code> — 예시 전처리 프로그램에서 성공한 단위 시험 개수입니다.</li><li><code>전체테스트수</code> — 실행하기로 한 단위 시험 전체 개수입니다. 기본 자료에서는 5개입니다.</li><li><code>처리오류수</code> — 전처리 중 예외 또는 잘못된 결과로 기록된 오류 건수입니다.</li></ul><h4>수행 순서</h4><ol><li>step2의 세 열을 확인하고 전체테스트수가 기본 자료에서 5인지 확인합니다.</li><li>통과테스트수와 전체테스트수가 같은지 비교합니다.</li><li>동시에 처리오류수가 0인지 확인합니다. 두 조건을 모두 만족해야 단위시험_충족=True로 표시합니다.</li><li>한 조건이라도 만족하지 않으면 False로 표시하고 step3에 저장합니다. 실패한 시험 이름이나 실행 오류는 현재 참고 코드 실행 로그에서 찾아 기록합니다.</li></ol><h4>결과를 읽는 방법</h4><p>단위시험_충족은 시험을 모두 통과했고 처리 오류도 없다는 두 조건의 결합입니다. 처리량과 체온 오차의 개선 여부는 이 열과 별도로 평가합니다.</p><h4>직접 확인할 가상 예시</h4><p>다음은 규칙을 이해하기 위한 가상 사례입니다. 실제 실행값을 대신 제출하는 자료가 아닙니다.</p><table><thead><tr><th>가정한 입력</th><th>기대 결과</th><th>이유</th></tr></thead><tbody><tr><td><code>{"통과테스트수": 5, "전체테스트수": 5, "처리오류수": 0}</code></td><td><code>True</code></td><td>모든 시험에 성공했고 처리 오류가 없습니다.</td></tr><tr><td><code>{"통과테스트수": 4, "전체테스트수": 5, "처리오류수": 0}</code></td><td><code>False</code></td><td>단위 시험 한 개가 실패했습니다.</td></tr><tr><td><code>{"통과테스트수": 5, "전체테스트수": 5, "처리오류수": 1}</code></td><td><code>False</code></td><td>시험은 통과했지만 처리 오류가 남았습니다.</td></tr></tbody></table><h4>완료 전 확인과 기록</h4><p>원행ID는 계산 전후 같은 자료를 찾는 번호입니다. 입력의 행 수·행 순서·인덱스·원래 열의 값을 유지하고, 앞 단계에서 만든 열도 모두 남깁니다. 각 행의 계산 결과는 새 열에 넣습니다. 실제 불리언 True·False와 문자열 "True"·"False"는 다릅니다. 값이 없다는 뜻의 결측값을 False나 0으로 대신 채우지 않습니다.</p><p>계산 뒤 해당 출력 열과 입력 열을 원행ID로 함께 확인합니다. 실제 자료에서 해당 사례가 있으면 최소 한 행의 입력·출력·판단 이유를 기록하고, 해당 사례가 없으면 0건 집계와 아래 가상 예시 시험 결과를 남깁니다. 관찰 내용은 5단계 완료 후 HTML의 해당 STEP 기록란에 옮기고 기록 JSON 저장을 눌러 보관합니다. HTML 입력란의 글은 페이지를 닫기 전에 저장해야 합니다.</p><h4>주의할 점</h4><p>전체테스트수가 0이면 시험을 하지 않은 것이므로 0/0을 성공으로 보고하지 않습니다. 제공된 5개 시험이 누락되었다면 기본 자료와 실행 상태를 교관에게 확인받습니다.</p>


> **교수자·해설용 FAIL 점검 포인트**: 통과 개수만 보거나, 처리 오류가 있는데도 True로 표시하거나, 시행하지 않은 시험을 통과로 세면 FAIL입니다.

In [ ]:
# M02 STEP 03 — 단위 시험 결과와 처리 중 오류를 함께 확인하기 (교수자·해설용 예시 풀이)
# 해설: 통과테스트수 == 전체테스트수 이고 처리오류수 == 0인 두 조건(AND)을 모두 만족해야 True입니다.
step3 = step2.copy()
step3["단위시험_충족"] = (
    (step3["통과테스트수"] == step3["전체테스트수"]) & (step3["처리오류수"] == 0)
).astype(bool)
display(step3[["원행ID", "단위시험_충족"]].head(10))

### M02 STEP 04 — 8분 고착의 시간 경계가 맞는지 확인하기

<h4>이 단계에서 해결할 일</h4><p>같은 값이 8개 이어지는 것과 8분 동안 이어지는 것은 다릅니다. 1분 간격으로 0분부터 7분까지 읽은 값은 8개지만 경과시간은 7분입니다. 0분부터 8분까지 9개가 있어야 8분 고착 기준을 시험할 수 있습니다.</p><h4>시작 전 준비</h4><p>각 미션의 STEP 1~5는 앞 단계 결과를 이어서 사용합니다. WEL01_m02_student.ipynb의 첫 실행(환경 준비) 셀과 이 미션의 준비 셀을 순서대로 실행하면 현재 참고 프로그램의 결과가 raw라는 입력 표로 만들어집니다. PARAMETERS는 기본값을 유지합니다. 이 단계만 다시 실행하려면 그 미션의 준비 셀부터 바로 앞 STEP까지 순서대로 실행해 둡니다.</p><p>이번 입력 표는 <code>step3</code>, 완성한 결과 표는 <code>step4</code>입니다. 새로 만들 열은 <code>시간경계_일치</code>이며 배정 시간은 24분입니다. 노트북의 <code>M02 STEP 04</code> 코드 셀에서 TODO 호출을 본인이 작성한 pandas 계산으로 바꾸세요. 결과는 DataFrame이어야 합니다.</p><h4>입력값의 뜻</h4><ul><li><code>고착8표본검출</code> — 같은 값 8개를 1분 간격으로 넣었을 때 고착이 검출되었는지 나타냅니다. 기본 8분 기준에서는 False여야 합니다.</li><li><code>고착9표본검출</code> — 같은 값 9개를 넣었을 때 검출되었는지 나타냅니다. 기본 기준에서는 True여야 합니다.</li></ul><h4>수행 순서</h4><ol><li>step3에서 두 경계 시험의 결과를 읽습니다. 기본 PARAMETERS의 FLATLINE_RUN이 8분인지 먼저 확인합니다.</li><li>고착8표본검출이 False인지 확인합니다.</li><li>고착9표본검출이 True인지 확인합니다. 두 조건을 모두 만족할 때만 시간경계_일치를 True로 표시합니다.</li><li>결과를 step4에 저장하고 8개·9개 관측의 첫 시각과 마지막 시각을 적어 경과시간을 설명합니다.</li></ol><h4>결과를 읽는 방법</h4><p>시간경계_일치=True는 기본 8분 기준의 양쪽 경계를 올바르게 처리했다는 뜻입니다. 실제 센서가 고장났다는 판단이나 사람의 건강 판단이 아닙니다.</p><h4>직접 확인할 가상 예시</h4><p>다음은 규칙을 이해하기 위한 가상 사례입니다. 실제 실행값을 대신 제출하는 자료가 아닙니다.</p><table><thead><tr><th>가정한 입력</th><th>기대 결과</th><th>이유</th></tr></thead><tbody><tr><td><code>{"고착8표본검출": false, "고착9표본검출": true}</code></td><td><code>True</code></td><td>7분 구간은 제외하고 8분 구간은 검출했습니다.</td></tr><tr><td><code>{"고착8표본검출": true, "고착9표본검출": true}</code></td><td><code>False</code></td><td>8개 관측을 8분으로 잘못 해석했습니다.</td></tr><tr><td><code>{"고착8표본검출": false, "고착9표본검출": false}</code></td><td><code>False</code></td><td>8분 구간도 검출하지 못했습니다.</td></tr></tbody></table><h4>완료 전 확인과 기록</h4><p>원행ID는 계산 전후 같은 자료를 찾는 번호입니다. 입력의 행 수·행 순서·인덱스·원래 열의 값을 유지하고, 앞 단계에서 만든 열도 모두 남깁니다. 각 행의 계산 결과는 새 열에 넣습니다. 실제 불리언 True·False와 문자열 "True"·"False"는 다릅니다. 값이 없다는 뜻의 결측값을 False나 0으로 대신 채우지 않습니다.</p><p>계산 뒤 해당 출력 열과 입력 열을 원행ID로 함께 확인합니다. 실제 자료에서 해당 사례가 있으면 최소 한 행의 입력·출력·판단 이유를 기록하고, 해당 사례가 없으면 0건 집계와 아래 가상 예시 시험 결과를 남깁니다. 관찰 내용은 5단계 완료 후 HTML의 해당 STEP 기록란에 옮기고 기록 JSON 저장을 눌러 보관합니다. HTML 입력란의 글은 페이지를 닫기 전에 저장해야 합니다.</p><h4>주의할 점</h4><p>기준을 4분으로 바꾸는 추가 실험에서는 기대 결과도 달라집니다. 기본 과제는 먼저 8분 설정으로 제출하고 변경 실험은 별도 기록합니다.</p>


> **교수자·해설용 FAIL 점검 포인트**: 두 시험 중 하나만 보거나, 관측 개수와 경과시간을 같은 숫자로 설명하면 FAIL입니다.

In [ ]:
# M02 STEP 04 — 8분 고착의 시간 경계가 맞는지 확인하기 (교수자·해설용 예시 풀이)
# 해설: 8표본(7분 경과)은 미검출(False), 9표본(8분 경과)은 검출(True)일 때만 시간경계_일치=True입니다.
step4 = step3.copy()
step4["시간경계_일치"] = (
    (~step4["고착8표본검출"].astype(bool)) & (step4["고착9표본검출"].astype(bool))
).astype(bool)
display(step4[["원행ID", "시간경계_일치"]].head(10))

### M02 STEP 05 — 전처리 결과에서 보완할 항목 찾기

<h4>이 단계에서 해결할 일</h4><p>오차가 줄었어도 속도가 부족하거나 시험이 실패하면 그대로 사용하도록 넘길 수 없습니다. 앞 단계에서 계산한 네 가지 결과를 한 행에서 확인하고, 어떤 조건 때문에 보완이 필요한지 구체적으로 남깁니다.</p><h4>시작 전 준비</h4><p>각 미션의 STEP 1~5는 앞 단계 결과를 이어서 사용합니다. WEL01_m02_student.ipynb의 첫 실행(환경 준비) 셀과 이 미션의 준비 셀을 순서대로 실행하면 현재 참고 프로그램의 결과가 raw라는 입력 표로 만들어집니다. PARAMETERS는 기본값을 유지합니다. 이 단계만 다시 실행하려면 그 미션의 준비 셀부터 바로 앞 STEP까지 순서대로 실행해 둡니다.</p><p>이번 입력 표는 <code>step4</code>, 완성한 결과 표는 <code>step5</code>입니다. 새로 만들 열은 <code>전처리_보완필요</code>이며 배정 시간은 24분입니다. 노트북의 <code>M02 STEP 05</code> 코드 셀에서 TODO 호출을 본인이 작성한 pandas 계산으로 바꾸세요. 결과는 DataFrame이어야 합니다.</p><h4>입력값의 뜻</h4><ul><li><code>RMSE_개선량</code> — 전처리 전 오차에서 후 오차를 뺀 값입니다. 0보다 커야 오차가 줄었습니다.</li><li><code>신호값_초당처리</code> — 실제 실행에서 계산한 신호값/초입니다. 목표는 1000 이상입니다.</li><li><code>단위시험_충족</code> — 시험 전부 통과와 처리 오류 0을 함께 확인한 값입니다.</li><li><code>시간경계_일치</code> — 8분 고착의 경계 시험이 맞는지 확인한 값입니다.</li></ul><h4>수행 순서</h4><ol><li>step4의 네 결과를 확인합니다. 이 단계에서 원래의 측정값이나 앞 단계 결과를 고치지 않습니다.</li><li>개선량이 0 이하, 처리량이 1000 미만, 단위시험_충족=False, 시간경계_일치=False 중 하나라도 있으면 전처리_보완필요=True로 만듭니다.</li><li>네 조건을 모두 만족한 경우에만 False로 표시하고 step5에 저장합니다. 처리량이 정확히 1000이면 속도 목표를 만족한 것입니다.</li><li>True인 상황마다 미달한 항목·현재 값·필요한 재시험을 기록합니다. False인 경우도 확인 범위가 오프라인 합성 실습이라는 점을 적습니다.</li></ol><h4>결과를 읽는 방법</h4><p>True는 전처리 방법이나 시험을 더 검토해야 한다는 뜻입니다. False는 이 네 가지 수치 조건에 따른 보완 사유가 없다는 뜻이며 실제 장비 적용 승인이 아닙니다.</p><h4>직접 확인할 가상 예시</h4><p>다음은 규칙을 이해하기 위한 가상 사례입니다. 실제 실행값을 대신 제출하는 자료가 아닙니다.</p><table><thead><tr><th>가정한 입력</th><th>기대 결과</th><th>이유</th></tr></thead><tbody><tr><td><code>{"RMSE_개선량": 0.02, "신호값_초당처리": 1000, "단위시험_충족": true, "시간경계_일치": true}</code></td><td><code>False</code></td><td>네 목표를 모두 충족합니다.</td></tr><tr><td><code>{"RMSE_개선량": 0, "신호값_초당처리": 2000, "단위시험_충족": true, "시간경계_일치": true}</code></td><td><code>True</code></td><td>오차 감소가 확인되지 않았습니다.</td></tr><tr><td><code>{"RMSE_개선량": 0.02, "신호값_초당처리": 999, "단위시험_충족": true, "시간경계_일치": true}</code></td><td><code>True</code></td><td>처리량이 목표보다 작습니다.</td></tr></tbody></table><h4>완료 전 확인과 기록</h4><p>원행ID는 계산 전후 같은 자료를 찾는 번호입니다. 입력의 행 수·행 순서·인덱스·원래 열의 값을 유지하고, 앞 단계에서 만든 열도 모두 남깁니다. 각 행의 계산 결과는 새 열에 넣습니다. 실제 불리언 True·False와 문자열 "True"·"False"는 다릅니다. 값이 없다는 뜻의 결측값을 False나 0으로 대신 채우지 않습니다.</p><p>계산 뒤 해당 출력 열과 입력 열을 원행ID로 함께 확인합니다. 실제 자료에서 해당 사례가 있으면 최소 한 행의 입력·출력·판단 이유를 기록하고, 해당 사례가 없으면 0건 집계와 아래 가상 예시 시험 결과를 남깁니다. 관찰 내용은 5단계 완료 후 HTML의 해당 STEP 기록란에 옮기고 기록 JSON 저장을 눌러 보관합니다. HTML 입력란의 글은 페이지를 닫기 전에 저장해야 합니다.</p><h4>주의할 점</h4><p>계산이 맞게 되어 True가 나온 것은 이 단계 코드의 실패가 아닙니다. 계산의 정확성과 예시 전처리 프로그램이 성능 목표를 달성했는지는 따로 평가합니다.</p>


> **교수자·해설용 FAIL 점검 포인트**: 네 항목이 모두 미달해야 True로 만들거나, 미달 항목을 숨기려고 값을 고치거나, True가 나왔다는 이유만으로 계산을 오답 처리하면 FAIL입니다.

In [ ]:
# M02 STEP 05 — 전처리 결과에서 보완할 항목 찾기 (교수자·해설용 예시 풀이)
# 해설: 개선량 <= 0, 처리량 < 1000, 단위시험_충족 == False, 시간경계_일치 == False 중 하나라도 있으면 True입니다.
step5 = step4.copy()
step5["전처리_보완필요"] = (
    (step5["RMSE_개선량"] <= 0)
    | (step5["신호값_초당처리"] < 1000)
    | (~step5["단위시험_충족"])
    | (~step5["시간경계_일치"])
).astype(bool)
display(step5[["원행ID", "전처리_보완필요"]].head(10))

### 마무리: 결과 화면 생성과 기록 저장

아래 셀을 실행하면 다섯 단계의 실제 계산 결과가 HTML로 저장됩니다. 왼쪽 STEP 메뉴로 결과를 확인하고 관찰 기록을 입력하세요. 기록 JSON을 내려받아야 입력한 글을 별도로 보관할 수 있습니다. 구조 검사 완료는 교관의 최종 합격 판정이 아닙니다.


In [ ]:
STAGE_SPECS = SOURCE_BUNDLE["missions"][MID]["stages"]
result_html = finish_mission(MID, raw, [step1, step2, step3, step4, step5], STAGE_SPECS)
